# Chapter 11 Sequence to Sequence Learning

# Sequence-to-Sequence Learning Part : 1

Sequence to sequence models are a variant of deep learning models that consists of an encoder and a decoder. They are used for problems that map an abitrarily long sequence to another arbitrarliy long sequence. For example, in machine translation, you convert a sequence of words in a source language to a sequence of words in a target language. Here we will see how we can use a seq2seq model to solve a machine translation task to convert English to German.


In [1]:
import random
import tensorflow as tf
import numpy as np
import time
import json

def fix_random_seed(seed):
    """ Setting the random seed of various libraries """
    try:
        np.random.seed(seed)
    except NameError:
        print("Warning: Numpy is not imported. Setting the seed for Numpy failed.")
    try:
        tf.random.set_seed(seed)
    except NameError:
        print("Warning: TensorFlow is not imported. Setting the seed for TensorFlow failed.")
    try:
        random.seed(seed)
    except NameError:
        print("Warning: random module is not imported. Setting the seed for random failed.")
 
# Fixing the random seed
random_seed=4321
fix_random_seed(random_seed)

print("TensorFlow version: {}".format(tf.__version__))

TensorFlow version: 2.20.0


http://www.manythings.org/anki/
    
german-english

In [2]:
# Not setting this led to the following error
# _Derived_]RecvAsync is cancelled.   
# [[{{node gradient_tape/model_1/embedding_1/embedding_lookup/Reshape/_172}}]] [Op:__inference_train_function_31985]

%env TF_FORCE_GPU_ALLOW_GROWTH=true

env: TF_FORCE_GPU_ALLOW_GROWTH=true


## Loading the data (Requires manual download)

Unfortunately, this dataset **must be manually downloaded** by clicking [this link](http://www.manythings.org/anki/deu-eng.zip). Then place the downloaded `deu-eng.zip` file in the `Ch11/data` folder before running the cell below.


In [6]:
# Section 11.1

import os
import requests
import zipfile

# Make sure the zip file has been downloaded
if not os.path.exists(os.path.join('data','deu-eng.zip')):
    raise FileNotFoundError(
        "Uh oh! Did you download the deu-eng.zip from http://www.manythings.org/anki/deu-eng.zip manually and place it in the Ch11/data folder?"
    )

else:
    if not os.path.exists(os.path.join('data', 'deu.txt')):
        with zipfile.ZipFile(os.path.join('data','deu-eng.zip'), 'r') as zip_ref:
            zip_ref.extractall('data')
    else:
        print("The extracted data already exists")

## Reading the data

Data is in a single `.txt` file. It is a parallel corpus meaning there is a English sentence/phrase/paragraph and a corresponding German translation of it side-by-side. In the file, the source input and the translation are separated by a tab (i.e. tab-seperated file)

In [7]:
# Section 11.1

import pandas as pd

# Read the csv file
df = pd.read_csv(os.path.join('data', 'deu.txt'), delimiter='\t', encoding='utf-8', encoding_errors="strict", header=None)
# Set column names
df.columns = ["EN", "DE", "Attribution"]
df = df[["EN", "DE"]]
print('df.shape = {}'.format(df.shape))

df.shape = (324282, 2)


In [8]:
# There are \xc2\xa0 (undecode-able bytes remaining in some text)
# This can cause errors like UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 3: unexpected end of data
# when using the TextVectorization layer
clean_inds = [i for i in range(len(df)) if b"\xc2" not in df.iloc[i]["DE"].encode("utf-8")]

df = df.iloc[clean_inds]

In [9]:
df.head()

,EN,DE
0,Go.,Geh.
1,Hi.,Hallo!
2,Hi.,Grüß Gott!
3,Run!,Lauf!
4,Run.,Lauf!


In [10]:
df.tail()

,EN,DE
324273,"When I was younger, I hated going to weddings....","Als ich jünger war, hasste ich es, auf Hochzei..."
324274,"It's so easy to write good example sentences, ...","Es ist so leicht, gute Beispielsätze zu schrei..."
324275,If someone who doesn't know your background sa...,"Wenn jemand, der deine Herkunft nicht kennt, s..."
324276,If someone who doesn't know your background sa...,"Wenn jemand Fremdes dir sagt, dass du dich wie..."
324278,If someone who doesn't know your background sa...,"Wenn einem von jemandem, der nicht weiß, woher..."


## Use a smaller sample for computational speed

There are more than 220000 samples in the original dataset. We will be using a smaller set of 50000 for our dataset. 

In [11]:
n_samples = 50000
df = df.sample(n=n_samples, random_state=random_seed)

## Introducing the `SOS` and `EOS` tokens (Decoder)

We will add these special tokens to the translated targets. `sos` indicates the start of the sentence and `eos` marks the end of the sentence. 

E.g. `Grüß Gott!` becomes `sos Grüß Gott! eos`

In [12]:
start_token = 'sos'
end_token = 'eos'

df["DE"] = start_token + ' ' + df["DE"] + ' ' + end_token

## Splitting training/validation/testing data

We will be creating three datasets by sampling randomly (without replacement);

* Test dataset - 5000 samples
* Validation dataset - 5000 samples
* Training dataset - 40000 samples

In [13]:
# Randomly sample 5000 examples from the total 50000 randomly
test_df = df.sample(n=int(n_samples/10), random_state=random_seed)
# Randomly sample 5000 examples from the total 50000 randomly
valid_df = df.loc[~df.index.isin(test_df.index)].sample(n=int(n_samples/10), random_state=random_seed)
# Assign the rest to training data
train_df = df.loc[~(df.index.isin(test_df.index) | df.index.isin(valid_df.index))]

print('test_df.shape = {}'.format(test_df.shape))
print('valid_df.shape = {}'.format(valid_df.shape))
print('train_df.shape = {}'.format(train_df.shape))

test_df.shape = (5000, 2)
valid_df.shape = (5000, 2)
train_df.shape = (40000, 2)


## Analysing the vocabulary sizes (English and German)

Calculate the vocabulary size. We will only consider the words that appear at least 10 times in the corpus.

In [14]:
# Section 11.1

from collections import Counter

# Create a flattened list from English words
en_words = train_df["EN"].str.split().sum()
# Create a flattened list of German words
de_words = train_df["DE"].str.split().sum()

# Get the vocabulary size of words appearing more than or equal to 10 times
n=10

# Code listing 11.1
def get_vocabulary_size_greater_than(words, n, verbose=True):
    
    """ Get the vocabulary size above a certain threshold """
    
    # Generate a counter object i.e. dict word -> frequency
    counter = Counter(words)
    
    # Create a pandas series from the counter, then sort most frequent to least
    freq_df = pd.Series(list(counter.values()), index=list(counter.keys())).sort_values(ascending=False)
    
    if verbose:
        # Print most common words
        print(freq_df.head(n=10))

    # Count of words >= n frequent    
    n_vocab = (freq_df>=n).sum()
    
    if verbose:
        print("\nVocabulary size (>={} frequent): {}".format(n, n_vocab))
        
    return n_vocab

print("English corpus")
print('='*50)
en_vocab = get_vocabulary_size_greater_than(en_words, n)

print("\nGerman corpus")
print('='*50)
de_vocab = get_vocabulary_size_greater_than(de_words, n)

English corpus
to      8842
Tom     8720
I       8682
you     8041
the     6669
a       5584
is      3924
that    2725
in      2544
of      2378
dtype: int64

Vocabulary size (>=10 frequent): 2173

German corpus
sos      40000
eos      40000
Tom       9184
Ich       8007
nicht     4771
ist       4360
Sie       3956
du        3551
zu        3540
das       3345
dtype: int64

Vocabulary size (>=10 frequent): 2453


## Analysing the sequence length (English and German)

Here we compute the sequence length of the sequences in the English and German corpora. To ignore the outliers, we only consider data between the 1% and 99% quantiles.

In [15]:
# Section 11.1

# Code listing 11.2
def print_sequence_length(str_ser):
    
    """ Print the summary stats of the sequence length """
    
    # Create a pd.Series, which contain the sequence length for each review
    seq_length_ser = str_ser.str.split(' ').str.len()

    # Get the median as well as summary statistics of the sequence length
    print("\nSome summary statistics")
    print("Median length: {}\n".format(seq_length_ser.median()))
    print(seq_length_ser.describe())
    
    # Get the quantiles at given marks
    print("\nComputing the statistics between the 1% and 99% quantiles (to ignore outliers)")
    p_01 = seq_length_ser.quantile(0.01)
    p_99 = seq_length_ser.quantile(0.99)
    
    # Print the summary stats of the data between the defined quantlies
    print(seq_length_ser[(seq_length_ser >= p_01) & (seq_length_ser < p_99)].describe())

print("English corpus")
print('='*50)
print_sequence_length(train_df["EN"])

print("\nGerman corpus")
print('='*50)
print_sequence_length(train_df["DE"])

English corpus

Some summary statistics
Median length: 6.0

count    40000.000000
mean         6.382925
std          2.505195
min          1.000000
25%          5.000000
50%          6.000000
75%          8.000000
max         44.000000
Name: EN, dtype: float64

Computing the statistics between the 1% and 99% quantiles (to ignore outliers)
count    39441.000000
mean         6.251946
std          2.229888
min          2.000000
25%          5.000000
50%          6.000000
75%          8.000000
max         13.000000
Name: EN, dtype: float64

German corpus

Some summary statistics
Median length: 8.0

count    40000.00000
mean         8.40480
std          2.51209
min          3.00000
25%          7.00000
50%          8.00000
75%         10.00000
max         53.00000
Name: DE, dtype: float64

Computing the statistics between the 1% and 99% quantiles (to ignore outliers)
count    39133.000000
mean         8.296067
std          2.186080
min          5.000000
25%          7.000000
50%          8.

## Printing the vocabulary size and sequence length

In [16]:
print("EN vocabulary size: {}".format(en_vocab))
print("DE vocabulary size: {}".format(de_vocab))

# Define sequence lengths with some extra space for longer sequences
en_seq_length = 19
de_seq_length = 21

print("EN max sequence length: {}".format(en_seq_length))
print("DE max sequence length: {}".format(de_seq_length))

EN vocabulary size: 2173
DE vocabulary size: 2453
EN max sequence length: 19
DE max sequence length: 21


## TensorFlow `TextVectorization` layer

The `TextVectorization` layer takes in strings and convert them to token IDs. The layer can build a vocabulary using a given text corups and uses that to generate the token IDs.

In [18]:
# Section 11.2

from tensorflow.keras.layers import TextVectorization

print("Defined the vectorization layer for English")

# Create the text vectorization layer (English)
en_vectorize_layer = TextVectorization(
    max_tokens=en_vocab,
    output_mode='int',
    output_sequence_length=None
)

print("Fitting the EN vectorization layer on data")
# Here we are calling adapt to fit the vectorization layer with text
# so that it learns the vocabulary
en_vectorize_layer.adapt(np.array(train_df["EN"].tolist()).astype('str'))
print("\tDone")

print("\nDefined the vectorization layer for German")

# Create the text vectorization layer (German)
de_vectorize_layer = TextVectorization(
    max_tokens=de_vocab,    
    output_mode='int',
    output_sequence_length=de_seq_length,
    pad_to_max_tokens=False,
)

print("Fitting the DE vectorization layer on data")
de_vectorize_layer.adapt(np.array(train_df["DE"].tolist()))
print("\tDone")

Defined the vectorization layer for English
Fitting the EN vectorization layer on data
	Done

Defined the vectorization layer for German
Fitting the DE vectorization layer on data
	Done


## `TextVectorization` layer in action
 
### How to use the layer (EN)

In [26]:
import tensorflow.keras.backend as K
K.clear_session()

# Create the model that uses the vectorize text layer
toy_model = tf.keras.models.Sequential()

# Start by creating an explicit input layer. It needs to have a shape of
# (1,) (because we need to guarantee that there is exactly one string
# input per batch), and the dtype needs to be 'string'.
toy_model.add(tf.keras.Input(shape=(1,), dtype=tf.string))

# The first layer in our model is the vectorization layer. After this
# layer, we have a tensor of shape (batch_size, max_len) containing vocab
# indices.
toy_model.add(en_vectorize_layer)

# Now, the model can map strings to integers, 
input_data = [["run"], ["I\'ll go home"],["ectoplasmic residue"]]
input_tensor = tf.constant(input_data)
pred = toy_model.predict(input_tensor)

print("Input data: \n{}\n".format(input_data))
print("\nToken IDs: \n{}".format(pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step
Input data: 
[['run'], ["I'll go home"], ['ectoplasmic residue']]


Token IDs: 
[[531   0   0]
 [ 79  46 104]
 [  1   1   0]]


### How to use the layer (DE)

In [27]:
import tensorflow.keras.backend as K
K.clear_session()

# Create the model that uses the vectorize text layer
toy_model = tf.keras.models.Sequential()

# Start by creating an explicit input layer. It needs to have a shape of
# (1,) (because we need to guarantee that there is exactly one string
# input per batch), and the dtype needs to be 'string'.
toy_model.add(tf.keras.Input(shape=(1,), dtype=tf.string))

# The first layer in our model is the vectorization layer. After this
# layer, we have a tensor of shape (batch_size, max_len) containing vocab
# indices.
toy_model.add(de_vectorize_layer)

# Now, the model can map strings to integers, 
input_data = [["[sos] Geh"], ["geh lauf"]]
input_tensor = tf.constant(input_data)
pred = toy_model.predict(input_tensor)

print("Input data: \n{}\n".format(input_data))
print("\nToken IDs: \n{}".format(pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Input data: 
[['[sos] Geh'], ['geh lauf']]


Token IDs: 
[[  2 642   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0]
 [642   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0]]


### Sample of the vocabulary

Let's print some words from the two vocabularies

In [21]:
# Section 11.2

print("English")
# Print first few words in the vocabulary
print(en_vectorize_layer.get_vocabulary()[:10])
# Print the size of the vocabulary
print(len(en_vectorize_layer.get_vocabulary()))

print("\nGerman")
# Print first few words in the vocabulary
print(de_vectorize_layer._lookup_layer.input_vocabulary)
print(de_vectorize_layer.get_vocabulary()[:10])

# Print the size of the vocabulary
print(len(de_vectorize_layer.get_vocabulary()))

English
['', '[UNK]', np.str_('you'), np.str_('tom'), np.str_('to'), np.str_('i'), np.str_('the'), np.str_('a'), np.str_('is'), np.str_('that')]
2173

German
None
['', '[UNK]', np.str_('sos'), np.str_('eos'), np.str_('ich'), np.str_('tom'), np.str_('nicht'), np.str_('ist'), np.str_('du'), np.str_('das')]
2453


## Defining the Seq2Seq model

Here we define an encoder decoder model to translate between English and German. We will be using a bidirectional encoder and a standard decoder. The model will use Gated Recurrent Unit (GRU) as the recurrent component. The encoder and the decoder has their own `TextVectorization` layers as they use two different languages. 

In [28]:
# Section 11.2

import tensorflow.keras.backend as K
K.clear_session()

# Code listing 11.3
def get_vectorizer(corpus, n_vocab, max_length=None, return_vocabulary=True, name=None):
    
    """ Return a text vectorization layer or a model """
    
    # Definie an input layer that takes a list of strings (or an array of strings)
    inp = tf.keras.Input(shape=(1,), dtype=tf.string, name='encoder_input')
    
    # When defining the vocab size, we'd add two for special tokens '' (Padding) and '[UNK]' (Oov tokens)
    vectorize_layer = tf.keras.layers.TextVectorization(
        max_tokens=n_vocab+2,
        output_mode='int',
        output_sequence_length=max_length,                
    )
    
    # Fit the vectorizer layer on the data
    vectorize_layer.adapt(corpus)
        
    # Get the token IDs
    vectorized_out = vectorize_layer(inp)
        
    if not return_vocabulary: 
        return tf.keras.models.Model(inputs=inp, outputs=vectorized_out, name=name)    
    else:
        # Returns the vocabulary in addition to the model
        return tf.keras.models.Model(inputs=inp, outputs=vectorized_out, name=name), vectorize_layer.get_vocabulary()
    
# Code listing 11.4   
def get_encoder(n_vocab, vectorizer):
    """ Define the encoder of the seq2seq model"""
    
    # The input is (None,1) shaped and accepts an array of strings
    inp = tf.keras.Input(shape=(1,), dtype=tf.string, name='e_input')

    # Vectorize the data (assign token IDs)
    vectorized_out = vectorizer(inp)
    
    # Define an embedding layer to convert IDs to word vectors
    emb_layer = tf.keras.layers.Embedding(n_vocab+2, 128, mask_zero=True, name='e_embedding')
    # Get the embeddings of the token IDs
    emb_out = emb_layer(vectorized_out)
    
    # Define a bidirectional GRU layer
    # Encoder looks at the english text (i.e. the input) both backwards and forward
    # this leads to better performance
    gru_layer = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(128, name='e_gru'), name='e_bidirectional_gru')
    
    # Get the output of the gru layer
    gru_out = gru_layer(emb_out)
    
    # Define the encoder model
    encoder = tf.keras.models.Model(inputs=inp, outputs=gru_out, name='encoder')
        
    return encoder


# Code listing 11.5
def get_final_seq2seq_model(n_vocab, encoder, vectorizer):
    """ Define the final encoder-decoder model """
    
    # Encoder's input
    e_inp = tf.keras.Input(shape=(1,), dtype=tf.string, name='e_input_final')    
    # Get the encoders final output
    d_init_state = encoder(e_inp)
    
    # The input is (None,1) shaped and accepts an array of strings
    # This input layer is used to train the seq2seq model with teacher-forcing
    # we feed the German sequence as the input and ask the model to predict 
    # it with the words offset by 1 (i.e. next word)
    d_inp = tf.keras.Input(shape=(1,), dtype=tf.string, name='d_input')
    
    # Vectorize the data (assign token IDs)
    d_vectorized_out = vectorizer(d_inp)
    
    # Define an embedding layer to convert IDs to word vectors
    # Note that this is a different embedding layer to the encoder's embedding layer
    d_emb_layer = tf.keras.layers.Embedding(n_vocab+2, 128, mask_zero=True, name='d_embedding')
    
    # Get the embeddings of the token IDs
    d_emb_out = d_emb_layer(d_vectorized_out)
    
    # Define a GRU layer
    # Unlike the encoder, we cannot define a bidirectional GRU for the decoder
    # Why?
    d_gru_layer = tf.keras.layers.GRU(256, return_sequences=True, name='d_gru')
    
    # Get the output of the gru layer
    d_gru_out = d_gru_layer(d_emb_out, initial_state=d_init_state)
    
    # Define an intermediate dense layer
    d_dense_layer_1 = tf.keras.layers.Dense(512, activation='relu', name='d_dense_1')
    d_dense1_out = d_dense_layer_1(d_gru_out)
    
    # The final prediction layer with softmax
    d_dense_layer_final = tf.keras.layers.Dense(n_vocab+2, activation='softmax', name='d_dense_final')
    d_final_out = d_dense_layer_final(d_dense1_out)
    
    # Define the full model
    seq2seq = tf.keras.models.Model(inputs=[e_inp, d_inp], outputs=d_final_out, name='final_seq2seq')
    
    return seq2seq

# Get the English vectorizer/vocabulary
en_vectorizer, en_vocabulary = get_vectorizer(np.array(train_df["EN"].tolist()), en_vocab, max_length=en_seq_length, name='e_vectorizer')
# Get the German vectorizer/vocabulary
de_vectorizer, de_vocabulary = get_vectorizer(np.array(train_df["DE"].tolist()), de_vocab, max_length=de_seq_length-1, name='d_vectorizer')

# Define the final model
encoder = get_encoder(en_vocab, en_vectorizer)
final_model = get_final_seq2seq_model(de_vocab, encoder, de_vectorizer)


## Compile the model

Compile the model with a suitable loss, an optimizer and metrics.

In [29]:
# Section 11.2
from tensorflow.keras.metrics import SparseCategoricalAccuracy

# Compile the model
final_model.compile(
    loss='sparse_categorical_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy']
)
final_model.summary()

Model: "final_seq2seq"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ d_input             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ d_vectorizer        │ (None, 20)        │          0 │ d_input[0][0]     │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ e_input_final       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ d_embedding         │ (None, 20, 128)   │    314,240 │ d_vectorizer[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder             │ (None, 256)       │    476,544 │ e_input_final[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ d_gru (GRU)         │ (None, 20, 256)   │    296,448 │ d_embedding[0][0… │
│                     │                   │            │ encoder[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ d_dense_1 (Dense)   │ (None, 20, 512)   │    131,584 │ d_gru[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ d_dense_final       │ (None, 20, 2455)  │  1,259,415 │ d_dense_1[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,478,231 (9.45 MB)

 Trainable params: 2,478,231 (9.45 MB)

 Non-trainable params: 0 (0.00 B)

## Evaluating MT models - BLEU metric

In machine translation, a popular choice for assessing performance is the BiLingual Evaluation Understudy (BLEU) metric. Word-to-word accuracy does not reflect the true performance of these models as there can be different ways the same phrase can be translated to. BLEU can take into account such multiple translations when computing the final score. Furthermore, BLEU is superior because it measures precision at multiple n-gram scales between the actual and predicted translations.

The implementation is inspired by: https://github.com/tensorflow/nmt/blob/master/nmt/scripts/bleu.py

### Defining the BLEU metric

Below we define a `BLEUMetric` object that can be used to compute the performance of the model.

In [30]:
# Section 11.3

from tensorflow.keras.layers import StringLookup
from bleu import compute_bleu

# Code listing 11.8
class BLEUMetric(object):
    
    def __init__(self, vocabulary, name='perplexity', **kwargs):
      """ Computes the BLEU score (Metric for machine translation) """
      super().__init__()
      self.vocab = vocabulary
      self.id_to_token_layer = StringLookup(vocabulary=self.vocab, num_oov_indices=0, oov_token="[KNU]", invert=True)
    
    def calculate_bleu_from_predictions(self, real, pred):
        """ Calculate the BLEU score for targets and predictions """
        
        # Get the predicted token IDs
        pred_argmax = tf.argmax(pred, axis=-1)  
        
        # Convert token IDs to words using the vocabulary and the StringLookup
        pred_tokens = self.id_to_token_layer(pred_argmax)
        real_tokens = self.id_to_token_layer(real)
        
        def clean_text(tokens):
            
            """ Clean padding and [SOS]/[EOS] tokens to only keep meaningful words """
            
            # 3. Strip the string of any extra white spaces
            translations_in_bytes = tf.strings.strip(
                        # 2. Replace everything after the eos token with blank
                        tf.strings.regex_replace(
                            # 1. Join all the tokens to one string in each sequence
                            tf.strings.join(
                                tf.transpose(tokens), separator=' '
                            ),
                        "eos.*", ""),
                   )
            
            # Decode the byte stream to a string
            translations = np.char.decode(
                translations_in_bytes.numpy().astype(np.bytes_), encoding='utf-8'
            )
            
            # If the string is empty, add a [UNK] token
            # Otherwise get a Division by zero error
            translations = [sent if len(sent)>0 else '[UNK]' for sent in translations ]
            
            # Split the sequences to individual tokens 
            translations = np.char.split(translations).tolist()
            
            return translations
        
        # Get the clean versions of the predictions and real seuqences
        pred_tokens = clean_text(pred_tokens)
        # We have to wrap each real sequence in a list to make use of a function to compute bleu
        real_tokens = [[token_seq] for token_seq in clean_text(real_tokens)]

        # The compute_bleu method accpets the translations and references in the following format
        # tranlation - list of list of tokens
        # references - list of list of list of tokens
        bleu, precisions, bp, ratio, translation_length, reference_length = compute_bleu(real_tokens, pred_tokens, smooth=False)

        return bleu

### Using the BLEU metric

Below you can see BLEU being used to computer the similarity between a translation (predicted) and reference (true target).

In [31]:
translation = [['[UNK]', '[UNK]', 'mÃssen', 'wir', 'in', 'erfahrung', 'bringen', 'wo', 'sie', 'wohnen']]
reference = [[['als', 'mÃssen', 'mÃssen', 'wir', 'in', 'erfahrung', 'bringen', 'wo', 'sie', 'wohnen']]]

bleu1, _, _, _, _, _ = compute_bleu(reference, translation)

translation = [['[UNK]', 'einmal', 'mÃssen', '[UNK]', 'in', 'erfahrung', 'bringen', 'wo', 'sie', 'wohnen']]
reference = [[['als', 'mÃssen', 'mÃssen', 'wir', 'in', 'erfahrung', 'bringen', 'wo', 'sie', 'wohnen']]]


bleu2, _, _, _, _, _ = compute_bleu(reference, translation)

print("BLEU score with longer correctly predicte phrases: {}".format(bleu1))
print("BLEU score without longer correctly predicte phrases: {}".format(bleu2))

BLEU score with longer correctly predicte phrases: 0.7598356856515925
BLEU score without longer correctly predicte phrases: 0.537284965911771


## Training the model with a custom loop

We will train the model using a custom loop as we want to incorporate BLEU as a metric in our training. We will follow the following procedure;

* Each epoch,
  * Shuffle the training data
  * Train our model on all the training data (in batches)
  * Evaluate the model on validation data
* Finally, evaluate the model on test data

In [35]:
# Section 11.3
import time

epochs = 5
batch_size = 128

# Code listing 11.6
def prepare_data(train_df, valid_df, test_df):
    """ Create a data dictionary from the dataframes containing data """
    
    data_dict = {}
    for label, df in zip(['train', 'valid', 'test'], [train_df, valid_df, test_df]):
        en_inputs = df["EN"].tolist()
        de_inputs = df["DE"].str.rsplit(n=1, expand=True).iloc[:,0].tolist()
        de_labels = df["DE"].str.split(n=1, expand=True).iloc[:,1].tolist()
        data_dict[label] = {'encoder_inputs': en_inputs, 'decoder_inputs': de_inputs, 'decoder_labels': de_labels}
    
    return data_dict

# Code listing 11.7
def shuffle_data(en_inputs, de_inputs, de_labels, shuffle_inds=None): 
    """ Shuffle the data randomly (but all of inputs and labels at ones)"""
        
    if shuffle_inds is None:
        # If shuffle_inds are not passed create a shuffling automatically
        shuffle_inds = np.random.permutation(np.arange(len(en_inputs)))
    else:
        # Shuffle the provided shuffle_inds
        shuffle_inds = np.random.permutation(shuffle_inds)
    
    # Return shuffled data - use list indexing
    en_shuffled = [en_inputs[i] for i in shuffle_inds]
    de_shuffled = [de_inputs[i] for i in shuffle_inds]
    de_labels_shuffled = [de_labels[i] for i in shuffle_inds]
    
    return (en_shuffled, de_shuffled, de_labels_shuffled), shuffle_inds


# Code listing 11.9
def evaluate_model(model, vectorizer, en_inputs_raw, de_inputs_raw, de_labels_raw, batch_size):
    """ Evaluate the model on various metrics such as loss, accuracy and BLEU """
    
    # Define the metric
    bleu_metric = BLEUMetric(de_vocabulary)
    
    loss_log, accuracy_log, bleu_log = [], [], []
    # Get the number of batches
    n_batches = len(en_inputs_raw)//batch_size
    print(" ", end='\r')

    # Evaluate one batch at a time
    for i in range(n_batches):
        # Status update
        print("Evaluating batch {}/{}".format(i+1, n_batches), end='\r')

        # Get the inputs and targets
        x = [tf.constant(en_inputs_raw[i*batch_size:(i+1)*batch_size]), tf.constant(de_inputs_raw[i*batch_size:(i+1)*batch_size])]
        y = vectorizer(tf.constant(de_labels_raw[i*batch_size:(i+1)*batch_size]))

        # Get the evaluation metrics
        loss, accuracy = model.evaluate(x, y, verbose=0)
        # Get the predictions to compute BLEU
        pred_y = model.predict(x, verbose=0)

        # Update logs
        loss_log.append(loss)
        accuracy_log.append(accuracy)
        bleu_log.append(bleu_metric.calculate_bleu_from_predictions(y, pred_y))
    
    return np.mean(loss_log), np.mean(accuracy_log), np.mean(bleu_log)
    

# Code listing 11.10
def train_model(model, vectorizer, train_df, valid_df, test_df, epochs, batch_size):
    """ Training the model and evaluating on validation/test sets """
    
    # Define the metric
    bleu_metric = BLEUMetric(de_vocabulary)

    # Define the data
    data_dict = prepare_data(train_df, valid_df, test_df)

    shuffle_inds = None
    
    
    for epoch in range(epochs):

        # Reset metric logs every epoch
        bleu_log = []
        accuracy_log = []
        loss_log = []

        # =================================================================== #
        #                         Train Phase                                 #
        # =================================================================== #

        # Shuffle data at the beginning of every epoch
        (en_inputs_raw,de_inputs_raw,de_labels_raw), shuffle_inds  = shuffle_data(
            data_dict['train']['encoder_inputs'],
            data_dict['train']['decoder_inputs'],
            data_dict['train']['decoder_labels'],
            shuffle_inds
        )

        # Get the number of training batches
        n_train_batches = len(en_inputs_raw)//batch_size

        # Train one batch at a time
        for i in range(n_train_batches):
            # Status update
            print("Training batch {}/{}".format(i+1, n_train_batches), end='\r')

            # Get a batch of inputs (english and german sequences)
            x = [tf.constant(en_inputs_raw[i*batch_size:(i+1)*batch_size]), tf.constant(de_inputs_raw[i*batch_size:(i+1)*batch_size])]
            # Get a batch of targets (german sequences offset by 1)
            y = vectorizer(tf.constant(de_labels_raw[i*batch_size:(i+1)*batch_size]))

            # Train for a single step
            model.train_on_batch(x, y)        
            # Evaluate the model to get the metrics
            loss, accuracy = model.evaluate(x, y, verbose=0)
            # Get the final prediction to compute BLEU
            pred_y = model.predict(x, verbose=0)

            # Update the epoch's log records of the metrics
            loss_log.append(loss)
            accuracy_log.append(accuracy)
            bleu_log.append(bleu_metric.calculate_bleu_from_predictions(y, pred_y))

        # =================================================================== #
        #                      Validation Phase                               #
        # =================================================================== #
        
        val_en_inputs = data_dict['valid']['encoder_inputs']
        val_de_inputs = data_dict['valid']['decoder_inputs']
        val_de_labels = data_dict['valid']['decoder_labels']
            
        val_loss, val_accuracy, val_bleu = evaluate_model(
            model, vectorizer, val_en_inputs, val_de_inputs, val_de_labels, batch_size
        )
            
        # Print the evaluation metrics of each epoch
        print("\nEpoch {}/{}".format(epoch+1, epochs))
        print("\t(train) loss: {} - accuracy: {} - bleu: {}".format(np.mean(loss_log), np.mean(accuracy_log), np.mean(bleu_log)))
        print("\t(valid) loss: {} - accuracy: {} - bleu: {}".format(val_loss, val_accuracy, val_bleu))
    
    # =================================================================== #
    #                      Test Phase                                     #
    # =================================================================== #    
    
    test_en_inputs = data_dict['test']['encoder_inputs']
    test_de_inputs = data_dict['test']['decoder_inputs']
    test_de_labels = data_dict['test']['decoder_labels']
            
    test_loss, test_accuracy, test_bleu = evaluate_model(
            model, vectorizer, test_en_inputs, test_de_inputs, test_de_labels, batch_size
    )
    
    print("\n(test) loss: {} - accuracy: {} - bleu: {}".format(test_loss, test_accuracy, test_bleu))


In [36]:

t1 = time.time()    
train_model(final_model, de_vectorizer, train_df, valid_df, test_df, epochs, batch_size)
t2 = time.time()

print("\nIt took {} seconds to complete the training".format(t2-t1))

Evaluating batch 39/39
Epoch 1/5
	(train) loss: 4.700072568196517 - accuracy: 0.2549998979442395 - bleu: 0.002490673457017475
	(valid) loss: 3.9258462404593444 - accuracy: 0.3345428070960901 - bleu: 0.0134858292568565
Evaluating batch 39/39
Epoch 2/5
	(train) loss: 3.535136830348235 - accuracy: 0.370918822594178 - bleu: 0.03262113188144644
	(valid) loss: 3.2698410780001907 - accuracy: 0.4041353036195804 - bleu: 0.05545050793330508
Evaluating batch 39/39
Epoch 3/5
	(train) loss: 2.9361226573968544 - accuracy: 0.4358956707784763 - bleu: 0.07450077422666553
	(valid) loss: 2.871076394350101 - accuracy: 0.449092162725253 - bleu: 0.08790271022000129
Evaluating batch 39/39
Epoch 4/5
	(train) loss: 2.496815485832019 - accuracy: 0.48552535923245627 - bleu: 0.11107956319938235
	(valid) loss: 2.5991885356414013 - accuracy: 0.4838567933975122 - bleu: 0.11326040870938324
Evaluating batch 39/39
Epoch 5/5
	(train) loss: 2.155180356059319 - accuracy: 0.5295014901038928 - bleu: 0.15052261712974394
	(va

## Save the trained model

We save the trained model as well as the vocabularies

In [40]:
# Section 11.3

## Save the model
os.makedirs('models', exist_ok=True)
# Note: Saving may fail on Windows with Unicode characters in vocabulary
# Uncomment below to save when needed
# tf.keras.models.save_model(final_model, os.path.join('models', 'seq2seq.keras'))

import json
os.makedirs(os.path.join('models', 'seq2seq_vocab'), exist_ok=True)

# Save the vocabulary files with UTF-8 encoding
with open(os.path.join('models', 'seq2seq_vocab', 'en_vocab.json'), 'w', encoding='utf-8') as f:
    json.dump(en_vocabulary, f, ensure_ascii=False)    
with open(os.path.join('models', 'seq2seq_vocab', 'de_vocab.json'), 'w', encoding='utf-8') as f:
    json.dump(de_vocabulary, f, ensure_ascii=False)

## Defining the inference model

For inference we have to create a new model using the weights of the trained model. During training we used teacher forcing, i.e. providing words from the translation as inputs to the decoder. This cannot be done during inference as we do not have a translation, but want to generate one.

Therefore, we create a decoder model that can generate one prediction at a time. We start the prediction process by giving the `sos` token as the initial input to the decoder and keep generating words until the decoder outputs `eos`.

In [48]:
# Section 11.4

# Code listing 11.11
import tensorflow.keras.backend as K
K.clear_session()

def get_inference_model_from_trained(model):
    """ Create an inference model from the trained model (without saving/loading) """
    
    # Get the encoder model
    en_model = model.get_layer("encoder")
    
    # Define two inputs
    # 1. Takes a single word as the input to the decoder
    d_inp = tf.keras.Input(shape=(1,), dtype=tf.string, name='d_infer_input')
    # 2. Takes an initial state to pass to the decoder GRU as an input
    d_state_inp = tf.keras.Input(shape=(256,), name='d_infer_state')
    
    # Create a new vectorizer with output_sequence_length=1 for single word inference
    trained_vectorizer = model.get_layer('d_vectorizer')
    d_vectorizer = tf.keras.layers.TextVectorization(
        max_tokens=de_vocab+2,    
        output_mode='int',
        output_sequence_length=1,  # Single word for inference
        pad_to_max_tokens=False,
    )
    # Copy vocabulary from trained vectorizer
    d_vectorizer.set_vocabulary(de_vectorize_layer.get_vocabulary())
    
    d_vectorized_out = d_vectorizer(d_inp)
    
    # Generate the embeddings from the vectorized input
    d_emb_layer = model.get_layer('d_embedding')
    d_emb_out = d_emb_layer(d_vectorized_out)
    
    # Create a NEW GRU layer with the same weights but different configuration
    trained_gru = model.get_layer("d_gru")
    d_gru_layer = tf.keras.layers.GRU(256, return_sequences=False, name='d_gru_inference')
    # Build the layer properly
    d_gru_layer.build(d_emb_out.shape)
    # Copy the weights from the trained model
    d_gru_layer.set_weights(trained_gru.get_weights())
    
    # Get the GRU out while using d_state_inp from earlier, as the initial state
    d_gru_out = d_gru_layer(d_emb_out, initial_state=d_state_inp) 
    
    # Get the dense output
    d_dense1_out = model.get_layer("d_dense_1")(d_gru_out) 
    
    # Get the final output
    d_final_out = model.get_layer("d_dense_final")(d_dense1_out) 
    
    # Define the final decoder
    de_model = tf.keras.models.Model(inputs=[d_inp, d_state_inp], outputs=[d_final_out, d_gru_out])
    
    return en_model, de_model

def get_vocabularies(save_dir):
    """ Load the vocabulary files from a given path"""
    
    with open(os.path.join(save_dir, 'en_vocab.json'), 'r', encoding='utf-8') as f:
        en_vocabulary = json.load(f)
        
    with open(os.path.join(save_dir, 'de_vocab.json'), 'r', encoding='utf-8') as f:
        de_vocabulary = json.load(f)
        
    return en_vocabulary, de_vocabulary

print("Loading vocabularies")
en_vocabulary, de_vocabulary = get_vocabularies(os.path.join('models', 'seq2seq_vocab'))

print("Creating inference model from trained model")
en_model, de_model = get_inference_model_from_trained(final_model)
print("\tDone")


Loading vocabularies
Creating inference model from trained model
	Done


## Generating new translations

Here we generate a new translation by first starting with the `sos` token and asking the decoder to generate words until it outputs `eos`.

In [49]:
# Code listing 11.12
def generate_new_translation(en_model, de_model, de_vocabulary, sample_en_text):
    """ Generate a new translation """
    
    start_token = 'sos'
    
    # Print the input
    print("Input: {}".format(sample_en_text))
    
    # Get the initial state for the decoder
    d_state = en_model.predict(tf.constant([[sample_en_text]]), verbose=0)
    d_state = tf.constant(d_state)  # Convert to tensor
    # First word will be sos
    de_word = start_token
    # We collect the translation in this list
    de_translation = []
    
    # Keep predicting until we get eos
    max_length = 50  # Safety limit
    for _ in range(max_length):
        if de_word == 'eos':
            break
        # Override the previous state input with the new state
        # Use __call__ instead of predict to maintain batch shape
        de_pred, d_state = de_model([tf.constant([[de_word]]), d_state], training=False)
        # Get the actual word from the token ID of the prediction
        de_word = de_vocabulary[np.argmax(de_pred[0].numpy())]
        # Add that to the translation
        de_translation.append(de_word)

    print("Translation: {}\n".format(' '.join(de_translation)))

for i in range(5):
    sample_en_text = test_df["EN"].iloc[i]
    generate_new_translation(en_model, de_model, de_vocabulary, sample_en_text)


Input: She pushed him out the door.
Translation: sie [UNK] ihn an die tür eos

Input: Tom doesn't use salt in his cooking.
Translation: tom [UNK] sich nicht auf die tür eos

Input: We're safe in here, aren't we?
Translation: wir sind hier nicht so [UNK] eos

Input: Is anybody thirsty?
Translation: ist jemand [UNK] eos

Input: You should have your head examined.
Translation: du hättest dir eine [UNK] [UNK] eos

